In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time
import json
import os

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
        self.c_proj = nn.Linear(d_model, d_model, bias=False)
        self.n_heads = n_heads
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        q = q.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)

        # Flash Attention is faster and saves memory
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return self.dropout(y)

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout_prob):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout_prob),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout_prob):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout_prob)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, dropout_prob)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config['vocab_size'], config['d_model']),
            wpe = nn.Embedding(config['max_seq_len'], config['d_model']),
            drop = nn.Dropout(config['dropout_prob']),
            h = nn.ModuleList([Block(config['d_model'], config['n_heads'], config['dropout_prob']) for _ in range(config['n_layers'])]),
            ln_f = nn.LayerNorm(config['d_model']),
        ))
        self.lm_head = nn.Linear(config['d_model'], config['vocab_size'], bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying

    def forward(self, idx, targets=None):
        B, T = idx.size()
        device = idx.device
        
        pos = torch.arange(0, T, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            # Flatten the logits and targets for cross entropy
            logits_flat = logits.view(-1, logits.size(-1))
            targets_flat = targets.view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
            
        return logits, loss

In [3]:
import numpy as np
bin_path = "/kaggle/input/notebooks/codingmaster24/tokenizer/corpus_tokens.bin"
data = np.memmap(bin_path, dtype=np.uint32, mode='r')

max_id = data.max()
print(f"The highest ID in your data is: {max_id}")
print(f"Your model vocab_size is: 32000")

if max_id >= 32000:
    print("CRITICAL: Your data has IDs that the model cannot handle!")

The highest ID in your data is: 36833
Your model vocab_size is: 32000
CRITICAL: Your data has IDs that the model cannot handle!


In [4]:
# 1. Update paths to explicitly match Kaggle's input structure
bin_path = "/kaggle/input/notebooks/codingmaster24/tokenizer/corpus_tokens.bin"
tokenizer_path = "/kaggle/input/notebooks/codingmaster24/tokenizer/tokenizer.json"

class MemmapDataset(Dataset):
    def __init__(self, bin_path, seq_len):
        # We read the file in 'r' mode. It does not load into RAM.
        self.data = np.memmap(bin_path, dtype=np.uint32, mode='r')
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) // self.seq_len - 1

    def __getitem__(self, idx):
        i = idx * self.seq_len
        chunk = self.data[i:i + self.seq_len + 1].astype(np.int64)
        x = torch.from_numpy(chunk[:-1])
        y = torch.from_numpy(chunk[1:])
        return x, y

print(f"Data ready. Total tokens to process: {len(np.memmap(bin_path, dtype=np.uint32, mode='r')):,}")

Data ready. Total tokens to process: 3,210,746,111


In [5]:
config = {
    "vocab_size": 37000,
    "d_model": 256,
    "n_heads": 8,
    "n_layers": 2, # For a 10M parameter model
    "max_seq_len": 512,
    "dropout_prob": 0.1,
    "architecture": "GPT-Causal-LM"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = GPT(config)
model.to(device)

# Enable Dual-GPU processing for T4x2
if torch.cuda.device_count() > 1:
    print(f"Success: Detected {torch.cuda.device_count()} GPUs. Wrapping model in DataParallel.")
    model = nn.DataParallel(model)

# Standard parameters
batch_size = 16 # Increased slightly because 2 GPUs handle 64 each
learning_rate = 3e-4
epochs = 1 

dataset = MemmapDataset(bin_path, seq_len=config["max_seq_len"])
# num_workers=2 prevents Kaggle CPU bottlenecks while feeding the GPUs
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = torch.amp.GradScaler('cuda') # For mixed precision training

Using device: cuda
Success: Detected 2 GPUs. Wrapping model in DataParallel.


In [6]:
print("Starting Training...")
model.train()

steps_per_epoch = len(dataloader)
log_interval = 100 
accumulation_steps = 8 # We will accumulate 4 small batches of 32 to get an effective batch size of 128

for epoch in range(epochs):
    start_time = time.time()
    optimizer.zero_grad(set_to_none=True) # Move zero_grad OUTSIDE the inner step loop
    
    for step, (x, y) in enumerate(dataloader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            logits, loss = model(x, y)
            if torch.cuda.device_count() > 1:
                loss = loss.mean()
            
            # Divide the loss by accumulation steps so the math scales correctly
            loss = loss / accumulation_steps
        
        # Accumulate the gradients
        scaler.scale(loss).backward()
        
        # Only update the weights every 4th step!
        if (step + 1) % accumulation_steps == 0 or (step + 1) == steps_per_epoch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True) # Reset gradients AFTER updating
        
        if step % log_interval == 0:
            elapsed = time.time() - start_time
            # Multiply loss by accumulation_steps just for printing the true value
            print(f"Epoch {epoch+1} | Step {step}/{steps_per_epoch} | Loss: {(loss.item() * accumulation_steps):.4f}")
            start_time = time.time()

print("Training Run Finished!")

Starting Training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 1 | Step 0/391937 | Loss: 10.6459
Epoch 1 | Step 100/391937 | Loss: 7.9778
Epoch 1 | Step 200/391937 | Loss: 5.8132
Epoch 1 | Step 300/391937 | Loss: 4.5858
Epoch 1 | Step 400/391937 | Loss: 3.8489
Epoch 1 | Step 500/391937 | Loss: 3.7532
Epoch 1 | Step 600/391937 | Loss: 3.6853
Epoch 1 | Step 700/391937 | Loss: 3.6547
Epoch 1 | Step 800/391937 | Loss: 3.8002
Epoch 1 | Step 900/391937 | Loss: 3.7237
Epoch 1 | Step 1000/391937 | Loss: 3.5809
Epoch 1 | Step 1100/391937 | Loss: 3.8020
Epoch 1 | Step 1200/391937 | Loss: 3.6633
Epoch 1 | Step 1300/391937 | Loss: 3.8600
Epoch 1 | Step 1400/391937 | Loss: 3.7453
Epoch 1 | Step 1500/391937 | Loss: 3.7854
Epoch 1 | Step 1600/391937 | Loss: 3.6895
Epoch 1 | Step 1700/391937 | Loss: 3.6667
Epoch 1 | Step 1800/391937 | Loss: 3.7400
Epoch 1 | Step 1900/391937 | Loss: 3.7453
Epoch 1 | Step 2000/391937 | Loss: 3.7187
Epoch 1 | Step 2100/391937 | Loss: 3.7124
Epoch 1 | Step 2200/391937 | Loss: 3.8248
Epoch 1 | Step 2300/391937 | Loss: 3.7754
Epo

In [7]:
# Save to Kaggle's working directory so you can download it
working_dir = "/kaggle/working/"
model_path = os.path.join(working_dir, "gpt_10m_model.pt")
config_path = os.path.join(working_dir, "config.json")

# Extract the base model state dict if it was wrapped in DataParallel
if isinstance(model, nn.DataParallel):
    state_dict = model.module.state_dict()
else:
    state_dict = model.state_dict()

torch.save(state_dict, model_path)
print(f"Model successfully saved to: {model_path}")

# Save the config file
with open(config_path, "w") as f:
    json.dump(config, f)
print(f"Config successfully saved to: {config_path}")

print("Check the 'Output' folder in your Kaggle right sidebar to download your final trained files!")

Model successfully saved to: /kaggle/working/gpt_10m_model.pt
Config successfully saved to: /kaggle/working/config.json
Check the 'Output' folder in your Kaggle right sidebar to download your final trained files!
